# ItemCF 调试 Notebook

一步一步调试 Recall_itemcf.py

In [3]:
# 导入必要的库
import os
import sys
import pickle
import random
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# 添加项目根目录到路径 (关键修复!)
project_root = Path(os.getcwd()).parent  # Notebook在Recall目录，父目录是项目根目录
sys.path.insert(0, str(project_root))
print(f"项目根目录: {project_root}")
print(f"Python路径: {sys.path[:2]}")

from util.utils import Logger, gen_user_item_time
from Recall.Recall_Methods import evaluate, itemcf_sim_cal_parallel, I2I_recall_parallel

warnings.filterwarnings('ignore')
random.seed(3)
print("✅ 导入成功！")

项目根目录: /home/pyy/XWTJ-main
Python路径: ['/home/pyy/XWTJ-main', '/root/miniconda3/envs/xwtj/lib/python39.zip']
✅ 导入成功！


In [4]:
# ==================== 路径配置 ====================
DATA_PATH = Path('../Initial_data')
SIM_MATRIX_PATH = Path('../Results/sim_matrix')
RECALL_DICT_PATH = Path('../Results/Recall_dict')
SUBMIT_PATH = Path('../Results/submit')
LOG_PATH = Path('../Results/log')

TRAIN_DATA_PATH = DATA_PATH / 'train_click_log.csv'
TEST_DATA_PATH = DATA_PATH / 'testA_click_log.csv'
ITEM_INFO_PATH = DATA_PATH / 'articles.csv'

print(f"训练数据路径: {TRAIN_DATA_PATH}")
print(f"测试数据路径: {TEST_DATA_PATH}")
print(f"物品信息路径: {ITEM_INFO_PATH}")

print(f"\n文件检查:")
print(f"  train_click_log.csv 存在: {TRAIN_DATA_PATH.exists()}")
print(f"  testA_click_log.csv 存在: {TEST_DATA_PATH.exists()}")
print(f"  articles.csv 存在: {ITEM_INFO_PATH.exists()}")

训练数据路径: ../Initial_data/train_click_log.csv
测试数据路径: ../Initial_data/testA_click_log.csv
物品信息路径: ../Initial_data/articles.csv

文件检查:
  train_click_log.csv 存在: True
  testA_click_log.csv 存在: True
  articles.csv 存在: True


## 步骤1：加载数据

In [5]:
# 加载训练数据
train_data = pd.read_csv(TRAIN_DATA_PATH)
print(f"训练数据形状: {train_data.shape}")
print(f"列: {train_data.columns.tolist()}")
train_data.head()

训练数据形状: (1112623, 9)
列: ['user_id', 'click_article_id', 'click_timestamp', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,199999,160417,1507029570190,4,1,17,1,13,1
1,199999,5408,1507029571478,4,1,17,1,13,1
2,199999,50823,1507029601478,4,1,17,1,13,1
3,199998,157770,1507029532200,4,1,17,1,25,5
4,199998,96613,1507029671831,4,1,17,1,25,5


In [7]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1112623 entries, 0 to 1112622
Data columns (total 9 columns):
 #   Column               Non-Null Count    Dtype
---  ------               --------------    -----
 0   user_id              1112623 non-null  int64
 1   click_article_id     1112623 non-null  int64
 2   click_timestamp      1112623 non-null  int64
 3   click_environment    1112623 non-null  int64
 4   click_deviceGroup    1112623 non-null  int64
 5   click_os             1112623 non-null  int64
 6   click_country        1112623 non-null  int64
 7   click_region         1112623 non-null  int64
 8   click_referrer_type  1112623 non-null  int64
dtypes: int64(9)
memory usage: 76.4 MB


In [8]:
# 加载测试数据
test_data = pd.read_csv(TEST_DATA_PATH)
print(f"测试数据形状: {test_data.shape}")
print(f"列: {test_data.columns.tolist()}")
test_data.head()

测试数据形状: (518010, 9)
列: ['user_id', 'click_article_id', 'click_timestamp', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type']


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,249999,160974,1506959142820,4,1,17,1,13,2
1,249999,160417,1506959172820,4,1,17,1,13,2
2,249998,160974,1506959056066,4,1,12,1,13,2
3,249998,202557,1506959086066,4,1,12,1,13,2
4,249997,183665,1506959088613,4,1,17,1,15,5


In [9]:
# 加载物品信息
item_info = pd.read_csv(ITEM_INFO_PATH)
item_info = item_info.rename(columns={'article_id': 'click_article_id'})  # 重命名物品ID列 为了合并
print(f"物品信息形状: {item_info.shape}")
item_info.head()  # 查看物品信息前五行  

物品信息形状: (364047, 4)


,click_article_id,category_id,created_at_ts,words_count
0,0,0,1513144419000,168
1,1,1,1405341936000,189
2,2,1,1408667706000,250
3,3,1,1408468313000,230
4,4,1,1407071171000,162


In [10]:
# 合并训练集和测试集
all_data = pd.concat([train_data, test_data], axis=0)
print(f"全量数据: {all_data.shape}")
print(f"用户数: {all_data['user_id'].nunique()}")
print(f"物品数: {all_data['click_article_id'].nunique()}")

全量数据: (1630633, 9)
用户数: 250000
物品数: 35380


## 步骤2：划分训练集和验证集

In [12]:
# 过滤只有一个点击的用户
df_filtered = all_data.groupby('user_id').filter(lambda x: len(x) > 1).reset_index(drop=True)
df_filtered = df_filtered.sort_values(by=['user_id', 'click_timestamp'], ascending=True)
print(f"过滤后: {df_filtered.shape}")
print(f"用户数: {df_filtered['user_id'].nunique()}")

过滤后: (1621113, 9)
用户数: 240480


In [18]:
# 训练历史：前n-1次点击
train_hist_df = df_filtered.groupby('user_id').apply(lambda x: x.iloc[:-1]).reset_index(drop=True)
print(f"训练历史: {train_hist_df.shape}")
train_hist_df.head(10)

训练历史: (1380633, 9)


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,30760,1508211672520,4,1,17,1,25,2
1,1,289197,1508211316889,4,1,17,1,25,6
2,2,36162,1508211438695,4,3,20,1,25,2
3,3,50644,1508211359672,4,3,2,1,25,2
4,4,42567,1508211625466,4,1,12,1,16,1
5,5,211442,1508211243884,4,4,2,1,25,2
6,6,62464,1508212140776,4,3,2,1,2,1
7,7,50644,1508211313878,4,1,17,1,25,2
8,8,70986,1508211172391,4,1,17,1,21,2
9,9,70986,1508211213614,4,3,2,1,25,2


In [17]:
# 验证标签：最后一次点击
train_last_df = df_filtered.groupby('user_id').tail(1).reset_index(drop=True)
print(f"验证标签: {train_last_df.shape}")
train_last_df.head(10)

验证标签: (240480, 9)


,user_id,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
0,0,157507,1508211702520,4,1,17,1,25,2
1,1,63746,1508211346889,4,1,17,1,25,6
2,2,168401,1508211468695,4,3,20,1,25,2
3,3,36162,1508211389672,4,3,2,1,25,2
4,4,39894,1508211655466,4,1,12,1,16,1
5,5,234481,1508211273884,4,4,2,1,25,2
6,6,10023,1508212170776,4,3,2,1,2,1
7,7,211442,1508211343878,4,1,17,1,25,2
8,8,50644,1508211202391,4,1,17,1,21,2
9,9,211455,1508211275547,4,3,2,1,25,2


## 步骤3：辅助数据

In [19]:
# 物品时间归一化
min_max_scale = lambda x: (x - np.min(x)) / (np.max(x) - np.min(x))
item_creat_ts_norm = item_info[['click_article_id', 'created_at_ts']].copy()
item_creat_ts_norm['created_at_ts'] = item_creat_ts_norm[['created_at_ts']].apply(min_max_scale)
item_creat_ts_norm_dict = dict(zip(item_creat_ts_norm['click_article_id'], item_creat_ts_norm['created_at_ts']))
print(f"时间字典大小: {len(item_creat_ts_norm_dict)}")
list(item_creat_ts_norm_dict.items())[:5]

时间字典大小: 364047


[(0, 0.9784319658749242),
 (1, 0.6802953033702287),
 (2, 0.6894929947449092),
 (3, 0.6889415569496703),
 (4, 0.6850776454577139)]

In [20]:
# 热门物品
hot_items = all_data['click_article_id'].value_counts().index[:350]
print(f"热门物品: {len(hot_items)}")

热门物品: 350


In [29]:
print("=" * 40)
print("数据统计")
print(f"训练用户: {train_hist_df['user_id'].nunique()}")
print(f"验证标签: {len(train_last_df)}")
print(f"测试用户: {test_data['user_id'].nunique()}")
print("=" * 40)

数据统计
训练用户: 240480
验证标签: 240480
测试用户: 50000
